In [3]:
from manim import *
from electrodynamics_classes import *
import jupyter_capture_output
from numpy import linalg as npl

video_scene = " -v WARNING --disable_caching charged_plane_Scene"
image_scene = f" -v WARNING --disable_caching -r {2*427},{2*240}  -s charged_plane_Scene"

Jupyter Capture Output v0.0.11


In [206]:
class ChargedPlate(Mobject):
	def __init__(self, plate_center, plate_normal, side_length, z_length, **kwargs):
		super().__init__(**kwargs)

		self.center = plate_center
		self.direction = plate_normal
		self.half_length = side_length / 2

		self.z_length = z_length

		# coordinate system
		self.e_x = UP
		self.e_z = self.direction / npl.norm(self.direction)
		self.e_y = -np.cross(self.e_z, self.e_x)

		x_arrow = Arrow3D(start = self.center, end = self.center + self.e_x, color = RED)
		y_arrow = Arrow3D(start = self.center, end = self.center + self.e_y, color = BLUE)
		z_arrow = Arrow3D(start = self.center, end = self.center + self.e_z, color = WHITE)
		# self.add(x_arrow, y_arrow, z_arrow)


		# plane
		plane_vertices = [
			self.center + self.half_length* (self.e_x+self.e_y),
			self.center + self.half_length* (self.e_x-self.e_y),
			self.center + self.half_length* (-self.e_x-self.e_y),
			self.center + self.half_length* (-self.e_x+self.e_y)
		]

		# plane
		plane = Polygon(*plane_vertices, color = WHITE, fill_color = WHITE, fill_opacity = 0.75, stroke_opacity = 0)
		self.add(plane)

		# horizontal axis
		horizontal_axis = DashedLine(start = self.center - 1.25*self.half_length*self.e_y, end = self.center + 1.25*self.half_length*self.e_y, color = RED)#)
		# self.add(horizontal_axis)


		# field point
		field_point_color = WHITE
		self.field_point_coord = self.center+z_length*self.e_z
		field_point_origin = Dot(self.center, radius = 0.03, color = field_point_color)
		field_point_arc = Arc(radius = 0.25, start_angle = 0, angle = -PI/2, arc_center = self.center)
		field_point_arc_dot = Dot(field_point_arc.get_center(), radius = 0.03, color = field_point_color).shift(0.025 * (UP+LEFT))
		field_point = Dot(self.field_point_coord, radius = 0.03, color = field_point_color)
		field_point_line = Line(start = self.center, end = self.field_point_coord, color = field_point_color)
		line_descriptor = MathTex("z", font_size = 24, color = field_point_color).next_to(field_point_line, 0.5*DOWN)
		# field_point_angle = 
		self.field_point_group = VGroup(field_point_origin, field_point, field_point_line, field_point_arc, field_point_arc_dot, line_descriptor)

		# radial circle


	# reference point dq on the upper half-plane
	def get_ref_point(self, r, phi):
		ref_point_color = RED
		x_ref = r * np.cos(phi) * self.e_x
		y_ref = r * np.sin(phi) * self.e_y
		ref_point_coord = self.center + x_ref + y_ref

		ref_point = Dot(ref_point_coord, radius = 0.03, color = ref_point_color)
		ref_point.r = Line(start = ref_point_coord, end = self.center, color = ref_point_color, stroke_width = 2)
		ref_point.connector = Line(start = ref_point_coord, end = self.field_point_coord, color = ref_point_color, stroke_width = 2)
		ref_point.phi = ArcBetweenPoints(start = self.center+0.65*self.e_z, end = self.center+0.65*(np.cos(phi) * self.e_x + np.sin(phi) * self.e_y), angle = phi, color = ref_point_color, stroke_width = 2)

		ref_point.dq_descriptor = MathTex(r"\mathrm{d}q", font_size = 24, color = ref_point_color).next_to(ref_point_coord, 0.4*UP)
		ref_point.r_descriptor = MathTex(r"r", font_size = 24, color = ref_point_color).next_to(ref_point.r, 0.1*LEFT).shift(0.1*(UP+RIGHT))
		ref_point.phi_descriptor = MathTex(r"\varphi", font_size = 24, color = ref_point_color).move_to(ref_point.phi.get_center()).shift(0.05 * (DOWN+1.2*LEFT))

		coulomb_dist = npl.norm(self.field_point_coord-ref_point_coord)
		dist = npl.norm(self.field_point_coord-self.center)
		alpha = np.arccos(dist / coulomb_dist)
		field_direction = (self.field_point_coord-ref_point_coord) / coulomb_dist
		ref_point.field_vector = Line(start = self.field_point_coord, end = self.field_point_coord + 2*field_direction, color = ref_point_color, stroke_width = 2).add_tip(tip_width = 0.1, tip_length = 0.1)
		ref_point.field_vector_descriptor = MathTex(r"\mathrm{d}\Vec{E}", font_size = 24, color = ref_point_color).next_to(ref_point.field_vector, RIGHT).shift(0.15*DOWN)
		ref_point.field_vector_z = Line(start = self.field_point_coord, end = self.field_point_coord + 2*np.cos(alpha)*self.e_z, color = BLUE, stroke_width = 2).add_tip(tip_width = 0.1, tip_length = 0.1)
		ref_point.field_vector_z_descriptor = MathTex(r"|\mathrm{d}\Vec{E}|\cos{(\alpha)}", font_size = 24, color = BLUE).next_to(ref_point.field_vector_z, RIGHT)#.shift(0.5*LEFT)
		ref_point.connector_alpha = ArcBetweenPoints(start = self.field_point_coord-2*field_direction, end = self.field_point_coord-2*self.e_z, angle = 4*alpha, color = ref_point_color, stroke_width = 2)
		ref_point.connector_alpha_descriptor = MathTex(r"\alpha", font_size = 24, color = ref_point_color).next_to(ref_point.connector_alpha.get_center(), 0.75*RIGHT)#.shift(0.05 * (DOWN+1.2*LEFT))
		ref_point.field_vector_z_connector = DashedLine(start = self.field_point_coord + 2*field_direction, end = self.field_point_coord + 2*np.cos(alpha)*self.e_z, color = ref_point_color, stroke_width = 2)
		return ref_point
	
	

	# equivalent of the ref point on the lower half-plane
	def get_shadow_ref_point(self, r, phi):
		shadow_ref_point_color = RED
		shadow_ref_point_opacity = 0.5
		x_ref = r * np.cos(phi+PI) * self.e_x
		y_ref = r * np.sin(phi+PI) * self.e_y
		shadow_ref_point_coord = self.center + x_ref + y_ref

		shadow_ref_point = Dot(shadow_ref_point_coord, radius = 0.03, color = shadow_ref_point_color, fill_opacity = shadow_ref_point_opacity)
		shadow_ref_point.r = Line(start = shadow_ref_point_coord, end = self.center, color = shadow_ref_point_color, stroke_opacity = shadow_ref_point_opacity, stroke_width = 2)
		shadow_ref_point.connector = Line(start = shadow_ref_point_coord, end = self.field_point_coord, color = shadow_ref_point_color, stroke_opacity = shadow_ref_point_opacity, stroke_width = 2)

		field_direction = (self.field_point_coord-shadow_ref_point_coord) / npl.norm(self.field_point_coord-shadow_ref_point_coord)
		shadow_ref_point.field_vector = Line(start = self.field_point_coord, end = self.field_point_coord + 2*field_direction, 
			stroke_opacity = shadow_ref_point_opacity, color = shadow_ref_point_color, stroke_width = 2).add_tip(tip_width = 0.1, tip_length = 0.1)
		
		coulomb_dist = npl.norm(self.field_point_coord-shadow_ref_point_coord)
		dist = npl.norm(self.field_point_coord-self.center)
		alpha = np.arccos(dist / coulomb_dist)
		shadow_ref_point.field_vector_z_connector = DashedLine(start = self.field_point_coord + 2*field_direction, end = self.field_point_coord + 2*np.cos(alpha)*self.e_z, 
			color = shadow_ref_point_color, stroke_width = 2, stroke_opacity = shadow_ref_point_opacity)
		return shadow_ref_point

In [ ]:
%%manim -qh --fps 60 $video_scene


class charged_plane_Scene(ThreeDScene):
	def construct(self):
		self.camera.background_color = inverted_main_color

		CVC = Text('CVC', font_size = 12, weight = BOLD, color = WHITE, font = 'Latin Modern Sans').align_on_border(RIGHT + DOWN, buff = 0.2)
		self.add(CVC)

		# headline
		headline = Title(r"Elektrisches Feld einer unendlich ausgedehnten Platte", font_size = 48, color = main_color).align_on_border(UP + LEFT, buff = 0.5).shift(0.5 * RIGHT)
		self.add(headline)

		# plate
		plate = ChargedPlate(plate_center = np.array([-4, 0, 0]), plate_normal = np.array([3, 0, 1]), side_length = 4, z_length = 6)
		self.add(plate)

		# ref point
		ref_point_r = 1.5
		ref_point_phi = PI/4


		ref_point = plate.get_ref_point(ref_point_r, ref_point_phi)
		shadow_ref_point = plate.get_shadow_ref_point(ref_point_r, ref_point_phi)
		# self.add(ref_point, shadow_ref_point)

		# self.add(ref_point.r, ref_point.connector, ref_point.phi, ref_point.dq_descriptor, ref_point.r_descriptor, ref_point.phi_descriptor, ref_point.field_vector, ref_point.field_vector_descriptor)
		# self.add(ref_point.field_vector_z, ref_point.field_vector_z_descriptor, ref_point.field_vector_z_connector, ref_point.connector_alpha, ref_point.connector_alpha_descriptor)
		# self.add(shadow_ref_point.r, shadow_ref_point.connector, shadow_ref_point.field_vector, shadow_ref_point.field_vector_z_connector)



		# def ref_point_updater(ref_point):
		# 	ref_point.become()


		# +++ ANIMATION +++
		self.wait(1.5)
		self.play(FadeIn(plate.field_point_group), run_time = 3)
		self.wait(1.5)
		self.play(FadeIn(ref_point), FadeIn(ref_point.dq_descriptor), run_time = 3)
		self.play(FadeIn(ref_point.r), FadeIn(ref_point.phi), FadeIn(ref_point.r_descriptor), FadeIn(ref_point.phi_descriptor), run_time = 3)
		self.wait(3)
		self.play(FadeIn(ref_point.connector), FadeIn(ref_point.field_vector), FadeIn(ref_point.connector_alpha), FadeIn(ref_point.connector_alpha_descriptor), FadeIn(ref_point.field_vector_descriptor), run_time = 3)
		self.wait(1.5)
		self.play(FadeIn(shadow_ref_point), FadeIn(shadow_ref_point.r), run_time = 3)
		self.play(FadeIn(shadow_ref_point.connector), FadeIn(shadow_ref_point.field_vector), run_time = 3)
		self.wait(3)
		self.play(FadeIn(ref_point.field_vector_z), FadeIn(ref_point.field_vector_z_descriptor), FadeIn(ref_point.field_vector_z_connector), FadeIn(shadow_ref_point.field_vector_z_connector), run_time = 3)
		self.wait(3)

Manim Community v0.18.1